# Ibovespa Pipeline — Observable / D3 / Vega-Lite

Pipeline 100% automático para baixar, processar e exportar dados do Ibovespa.

| Passo | O que faz |
|-------|-----------|
| 1 | Composição do Ibovespa (API B3) |
| 2 | Validação dos tickers no Yahoo Finance |
| 3 | Download histórico OHLCV 2018–2025 |
| 4 | Cálculo de derivados (retorno, volatilidade, drawdown…) |
| 5 | Consolidação e export de CSVs |
| 6 | Grafo temporal de correlações (JSON para D3) |
| 7 | Resumo por ação — scatter Risco × Retorno |
| 8 | Resumo por ação × ano |
| 9 | **[NOVO]** Taxa Selic histórica (API Banco Central) |
| 10 | **[NOVO]** Setor e indústria de cada ação (Yahoo Finance) |

## Imports

In [ ]:
import requests
import json
import base64
import time
import os
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm

warnings.filterwarnings("ignore")
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import LabelEncoder
import networkx as nx
try:
    import community as community_louvain
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-louvain', 'networkx', 'scikit-learn', '-q'])
    import community as community_louvain


## Configurações

In [ ]:
OUTPUT_DIR = "ibovespa_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = "2018-01-01"
END_DATE   = "2025-12-31"

# Limiar de correlação para gerar aresta no grafo
THRESHOLD = 0.5

# Períodos históricos relevantes para o grafo de correlações
PERIODS = [
    ("pre_covid",        "2018-01-01", "2019-12-31"),
    ("covid_crash",      "2020-01-01", "2020-06-30"),
    ("covid_recovery",   "2020-07-01", "2020-12-31"),
    ("post_covid",       "2021-01-01", "2021-12-31"),
    ("alta_selic",       "2022-01-01", "2022-12-31"),
    ("eleicoes_out2022", "2022-10-01", "2022-12-31"),
    ("2023",             "2023-01-01", "2023-12-31"),
    ("2024",             "2024-01-01", "2024-12-31"),
    ("2025",             "2025-01-01", "2025-12-31"),
]

print(f"Pasta de saída : {OUTPUT_DIR}/")
print(f"Período        : {START_DATE} → {END_DATE}")
# Períodos trimestrais para H4 (comunidades)
import itertools
quarters = []
for year in range(2018, 2026):
    for q, (m0, m1) in enumerate([(1,3),(4,6),(7,9),(10,12)], 1):
        start_q = f"{year}-{m0:02d}-01"
        end_q   = f"{year}-{m1:02d}-{[31,28,31,30,31,30,31,31,30,31,30,31][m1-1]:02d}"
        if start_q <= END_DATE:
            quarters.append((f"{year}Q{q}", start_q, end_q))

PERIODS_QUARTERLY = quarters
print(f"Períodos trimestrais gerados: {len(PERIODS_QUARTERLY)}")


## Passo 1 — Composição do Ibovespa (API B3)

In [ ]:
def build_b3_url(page: int = 1, page_size: int = 120) -> str:
    payload = {
        "language":   "pt-br",
        "pageNumber": page,
        "pageSize":   page_size,
        "index":      "IBOV",
        "segment":    "1",
    }
    encoded = base64.b64encode(json.dumps(payload).encode()).decode()
    return (
        "https://sistemaswebb3-listados.b3.com.br"
        f"/indexProxy/indexCall/GetPortfolioDay/{encoded}"
    )


def fetch_ibov_composition() -> pd.DataFrame:
    """
    Retorna DataFrame com colunas:
      cod  (ticker B3, ex: 'PETR4')
      asset, type, part (participação %)
    """
    print("🔍 Buscando composição do Ibovespa na API da B3...")
    all_rows = []
    page = 1

    while True:
        url = build_b3_url(page=page)
        try:
            resp = requests.get(url, timeout=15,
                                headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"   ⚠️  Erro na página {page}: {e}")
            break

        results = data.get("results", [])
        if not results:
            break

        all_rows.extend(results)

        total = data.get("header", {}).get("totalRecords", 0)
        if len(all_rows) >= total or len(results) == 0:
            break

        page += 1
        time.sleep(0.3)

    df = pd.DataFrame(all_rows)
    df["cod"] = df["cod"].str.strip()
    df["ticker_yf"] = df["cod"] + ".SA"

    print(f"   {len(df)} ativos encontrados na carteira do Ibovespa.")
    print(f"   Exemplos: {df['cod'].head(8).tolist()}")
    return df


composition = fetch_ibov_composition()

comp_path = os.path.join(OUTPUT_DIR, "ibovespa_composicao_b3.csv")
composition.to_csv(comp_path, index=False)
print(f"    Composição salva: {comp_path}")

TICKERS_YF = composition["ticker_yf"].tolist()
print(f"\n Total de tickers para download: {len(TICKERS_YF)}")

## Passo 2 — Validar tickers no Yahoo Finance

In [ ]:
def validate_tickers(tickers: list, sample_days: int = 5) -> tuple:
    """Retorna (tickers_validos, tickers_invalidos)."""
    print(f"🔎 Validando {len(tickers)} tickers no Yahoo Finance...")

    valid, invalid = [], []
    probe = yf.download(
        tickers,
        period=f"{sample_days}d",
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True,
    )

    for tk in tickers:
        try:
            col = probe[tk]["Close"] if len(tickers) > 1 else probe["Close"]
            if col.dropna().empty:
                invalid.append(tk)
            else:
                valid.append(tk)
        except Exception:
            invalid.append(tk)

    print(f"    Válidos  : {len(valid)}")
    if invalid:
        print(f"    Inválidos: {len(invalid)} → {[t.replace('.SA','') for t in invalid]}")

    return valid, invalid


valid_tickers, invalid_tickers = validate_tickers(TICKERS_YF)

## Passo 3 — Download histórico completo (2018–2025)

`auto_adjust=True` → preços já corrigidos por splits e dividendos.

In [ ]:
print(f"📥 Baixando histórico {START_DATE} → {END_DATE} "
      f"para {len(valid_tickers)} tickers...\n")

raw = yf.download(
    valid_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    group_by="ticker",
    threads=True,
    progress=True,
)

print(f"\n Download concluído.")

## Passo 4 — Processamento e cálculo de derivados

Por ticker extraímos OHLCV ajustado e calculamos:
- `return_daily` — retorno % diário
- `log_return` — log-retorno (base para correlação de Pearson)
- `volatility_20d` — volatilidade rolling 20 dias, anualizada (%)
- `amplitude_intraday` — (High−Low)/Open×100 (proxy de vol intradia)
- `cumulative_return` — retorno acumulado desde início da série (%)

In [ ]:
all_frames = []
failed     = []

for tk in tqdm(valid_tickers, desc="Processando tickers"):
    try:
        df = (raw[tk] if len(valid_tickers) > 1 else raw).copy()
        df.dropna(subset=["Close"], inplace=True)

        if len(df) < 50:
            failed.append(tk); continue

        # Preenche gaps pontuais (feriados isolados)
        df = df.interpolate(method="time", limit=5)

        df["return_daily"]       = df["Close"].pct_change() * 100 # remover vezes 100
        df["log_return"]         = np.log(df["Close"] / df["Close"].shift(1))
        df["volatility_20d"]     = (
            df["log_return"].rolling(20).std() * np.sqrt(252) * 100
        )
        df["amplitude_intraday"] = (df["High"] - df["Low"]) / df["Open"] * 100 # remover vezes 100
        df["cumulative_return"]  = (
            (1 + df["return_daily"] / 100).cumprod() - 1
        ) * 100

        df["ticker"] = tk
        df.index.name = "date"
        df.reset_index(inplace=True)
        all_frames.append(df)

    except Exception as e:
        failed.append(tk)
        print(f"   ⚠️  {tk}: {e}")

print(f"\n Processados: {len(all_frames)} |  Falhas: {len(failed)}")
if failed:
    print(f"   {[t.replace('.SA','') for t in failed]}")

## Passo 5 — Consolidação e export de CSVs

In [ ]:
COLS = [
    "date", "ticker",
    "Open", "High", "Low", "Close", "Volume",
    "return_daily", "log_return",
    "volatility_20d", "amplitude_intraday",
    "cumulative_return",
]

df_all = pd.concat(all_frames, ignore_index=True)[COLS]
float_cols = [c for c in COLS if c not in ("date", "ticker")]
df_all[float_cols] = df_all[float_cols].round(6)

print(f"📊 Dataset final:")
print(f"   Linhas : {len(df_all):,}")
print(f"   Ações  : {df_all['ticker'].nunique()}")
print(f"   Período: {df_all['date'].min()} → {df_all['date'].max()}")

# 5a. CSV long format (principal)
path_long = os.path.join(OUTPUT_DIR, "ibovespa_historico_2018_2025.csv")
df_all.to_csv(path_long, index=False)

# 5b. Pivot de fechamentos (linhas=datas, colunas=tickers)
df_close = df_all.pivot(index="date", columns="ticker", values="Close")
df_close.to_csv(os.path.join(OUTPUT_DIR, "ibovespa_close_pivot.csv"))

# 5c. Pivot de log-retornos (base para correlações)
df_logret = df_all.pivot(index="date", columns="ticker", values="log_return")
df_logret.to_csv(os.path.join(OUTPUT_DIR, "ibovespa_logreturn_pivot.csv"))

print(f"\n CSVs salvos em: {OUTPUT_DIR}/")

## Passo 6 — Setor e indústria de cada ação (Yahoo Finance)

Baixa `sector` e `industry` via `yf.Ticker().info` para cada ação do Ibovespa.
O CSV gerado pode ser usado como tabela de lookup no Observable para colorir/filtrar por setor.


In [ ]:
def fetch_setores(tickers_yf: list) -> pd.DataFrame:
    """
    Retorna DataFrame com colunas:
      ticker_yf, ticker_clean, longName, shortName, sector, industry
    """
    print(f" Buscando setor/indústria para {len(tickers_yf)} ações...")
    rows = []

    for tk in tqdm(tickers_yf, desc="Baixando setores"):
        row = {
            "ticker_yf":    tk,
            "ticker_clean": tk.replace(".SA", ""),
            "longName":     "",
            "shortName":    "",
            "sector":       "",
            "industry":     "",
        }
        try:
            info = yf.Ticker(tk).info
            row["longName"]  = info.get("longName", "")
            row["shortName"] = info.get("shortName", "")
            row["sector"]    = info.get("sector", "")
            row["industry"]  = info.get("industry", "")
            time.sleep(0.2)   # respeita rate-limit do Yahoo
        except Exception as e:
            print(f"     {tk}: {e}")

        rows.append(row)

    return pd.DataFrame(rows)


df_setores = fetch_setores(valid_tickers)

setores_path = os.path.join(OUTPUT_DIR, "ibovespa_setores.csv")
df_setores.to_csv(setores_path, index=False)

print(f"\n Distribuição por setor:")
print(
    df_setores[df_setores["sector"] != ""]
    .groupby("sector")["ticker_clean"]
    .count()
    .sort_values(ascending=False)
    .to_string()
)
print(f"\n Setores salvos: {setores_path}")

## Passo 7 — Grafo temporal de correlações por período

Calcula a matriz de correlação de Pearson entre log-retornos por período histórico e exporta nós + arestas em JSON para uso direto no D3 force-directed.

**Limiar:** `|correlação| >= THRESHOLD` → vira aresta no grafo.


In [ ]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import community as community_louvain  # python-louvain

print("🔗 Calculando grafos de correlação por período...")

def build_graph_for_period(name, p0, p1, df_logret, df_all, df_setores, threshold=THRESHOLD):
    """Constrói grafo de correlação, detecta comunidades (Louvain) e calcula
    modularidade + NMI/ARI em relação aos setores reais."""
    sub = df_logret.loc[p0:p1].dropna(axis=1, thresh=30)
    tks = sub.columns.tolist()
    if len(tks) < 5:
        return None
    corr = sub.corr(method="pearson")

    # ── Nós com setor
    setor_lookup = df_setores.set_index("ticker_yf")[["sector", "industry", "longName"]].to_dict("index")
    period_df = df_all[(df_all["date"] >= p0) & (df_all["date"] <= p1)]
    nodes = []
    for tk in tks:
        t = period_df[period_df["ticker"] == tk]
        if t.empty:
            continue
        info = setor_lookup.get(tk, {})
        nodes.append({
            "id":             tk,
            "ticker_clean":   tk.replace(".SA", ""),
            "avg_close":      round(float(t["Close"].mean()), 2),
            "avg_volume":     round(float(t["Volume"].mean()), 0),
            "avg_volatility": round(float(t["volatility_20d"].mean()), 4),
            "total_return":   round(float(t["cumulative_return"].iloc[-1]), 4),
            "avg_amplitude":  round(float(t["amplitude_intraday"].mean()), 4),
            "sector":         info.get("sector", "Unknown"),
            "industry":       info.get("industry", "Unknown"),
            "longName":       info.get("longName", ""),
        })

    # ── Arestas
    node_ids = {n["id"] for n in nodes}
    edges = [
        {
            "source":      tks[i],
            "target":      tks[j],
            "correlation": round(float(corr.iloc[i, j]), 4),
            "weight":      round(abs(float(corr.iloc[i, j])), 4),
        }
        for i in range(len(tks))
        for j in range(i + 1, len(tks))
        if tks[i] in node_ids and tks[j] in node_ids
        and not np.isnan(corr.iloc[i, j])
        and abs(corr.iloc[i, j]) >= threshold
    ]

    # ── Detecção de comunidades (Louvain) sobre grafo positivo
    G = nx.Graph()
    G.add_nodes_from([n["id"] for n in nodes])
    for e in edges:
        if e["correlation"] > 0:  # só correlações positivas para Louvain
            G.add_edge(e["source"], e["target"], weight=e["weight"])

    partition = {}
    modularity = None
    nmi = None
    ari = None

    if G.number_of_edges() > 0:
        try:
            partition = community_louvain.best_partition(G, weight="weight")
            modularity = round(community_louvain.modularity(partition, G), 4)

            # Comparar comunidades detectadas vs setor real
            common = [n["id"] for n in nodes if n["id"] in partition and n["sector"] not in ("", "Unknown")]
            if len(common) >= 10:
                detected = [partition[tk] for tk in common]
                sector_labels = [next(n["sector"] for n in nodes if n["id"] == tk) for tk in common]
                le = LabelEncoder()
                sector_enc = le.fit_transform(sector_labels)
                nmi = round(float(normalized_mutual_info_score(sector_enc, detected)), 4)
                ari = round(float(adjusted_rand_score(sector_enc, detected)), 4)
        except Exception as ex:
            print(f"   ⚠️  Louvain falhou em {name}: {ex}")

    # Anotar comunidade em cada nó
    for n in nodes:
        n["community"] = partition.get(n["id"], -1)

    return {
        "period":     name,
        "start":      p0,
        "end":        p1,
        "n_nodes":    len(nodes),
        "n_edges":    len(edges),
        "modularity": modularity,
        "nmi_sector": nmi,   # NMI comunidade vs setor (0=ruim, 1=perfeito)
        "ari_sector": ari,   # ARI (0=aleatório, 1=perfeito)
        "nodes":      nodes,
        "edges":      edges,
    }


# ── Períodos temáticos (H1/H2)
graph_data = []
for name, p0, p1 in PERIODS:
    result = build_graph_for_period(name, p0, p1, df_logret, df_all, df_setores)
    if result:
        graph_data.append(result)
        print(f"   {name:22s} → nós: {result['n_nodes']:3d} | arestas: {result['n_edges']:5d} "
              f"| mod: {result['modularity']} | NMI: {result['nmi_sector']} | ARI: {result['ari_sector']}")

graph_path = os.path.join(OUTPUT_DIR, "ibovespa_correlation_graph.json")
with open(graph_path, "w", encoding="utf-8") as f:
    json.dump(graph_data, f, ensure_ascii=False, default=str)
print(f"\n💾 Grafo temático salvo: {graph_path}")


# ── Períodos trimestrais (H4)
print("\n🔗 Calculando grafos TRIMESTRAIS para H4...")
graph_quarterly = []
for name, p0, p1 in PERIODS_QUARTERLY:
    result = build_graph_for_period(name, p0, p1, df_logret, df_all, df_setores)
    if result:
        graph_quarterly.append(result)
        print(f"   {name} → nós: {result['n_nodes']:3d} | arestas: {result['n_edges']:5d} "
              f"| mod: {result['modularity']} | NMI: {result['nmi_sector']}")

quarterly_path = os.path.join(OUTPUT_DIR, "ibovespa_correlation_graph_quarterly.json")
with open(quarterly_path, "w", encoding="utf-8") as f:
    json.dump(graph_quarterly, f, ensure_ascii=False, default=str)
print(f"\n💾 Grafo trimestral salvo: {quarterly_path}")


# ── Tabela resumo de modularidade e NMI por período (trimestral)
df_h4 = pd.DataFrame([
    {
        "period":     g["period"],
        "start":      g["start"],
        "end":        g["end"],
        "n_nodes":    g["n_nodes"],
        "n_edges":    g["n_edges"],
        "modularity": g["modularity"],
        "nmi_sector": g["nmi_sector"],
        "ari_sector": g["ari_sector"],
    }
    for g in graph_quarterly
])
h4_path = os.path.join(OUTPUT_DIR, "h4_community_vs_sector.csv")
df_h4.to_csv(h4_path, index=False)
print(f"💾 Tabela H4 salva: {h4_path}")
print("\n", df_h4.to_string(index=False))


## Passo 8 — Resumo por ação (scatter Risco × Retorno)


In [ ]:
rows = []
for tk in df_all["ticker"].unique():
    d = df_all[df_all["ticker"] == tk].dropna(subset=["Close"])
    if len(d) < 100: continue
    rows.append({
        "ticker":              tk,
        "ticker_clean":        tk.replace(".SA", ""),
        "n_days":              len(d),
        "first_date":          str(d["date"].min()),
        "last_date":           str(d["date"].max()),
        "price_start":         round(float(d.iloc[0]["Close"]), 2),
        "price_end":           round(float(d.iloc[-1]["Close"]), 2),
        "total_return_pct":    round(float(d["cumulative_return"].iloc[-1]), 4),
        "avg_daily_return":    round(float(d["return_daily"].mean()), 6),
        "std_daily_return":    round(float(d["return_daily"].std()), 6),
        "avg_volatility_20d":  round(float(d["volatility_20d"].mean()), 4),
        "max_drawdown_pct":    round(
            float(((d["Close"] / d["Close"].cummax()) - 1).min() * 100), 4
        ),
        "avg_volume":          round(float(d["Volume"].mean()), 0),
        "avg_amplitude_intraday": round(float(d["amplitude_intraday"].mean()), 4),
    })

df_summary = pd.DataFrame(rows)
summary_path = os.path.join(OUTPUT_DIR, "ibovespa_summary_por_acao.csv")
df_summary.to_csv(summary_path, index=False)

print(f"💾 Resumo por ação salvo: {summary_path}")
print("\n🏆 Top 10 por retorno acumulado (2018–2025):")
print(
    df_summary[["ticker_clean", "total_return_pct",
                "avg_volatility_20d", "max_drawdown_pct"]]
    .sort_values("total_return_pct", ascending=False)
    .head(10)
    .to_string(index=False)
)

## Passo 9 — Resumo por ação × ano


In [ ]:
rows = []
for tk in df_all["ticker"].unique():
    d = df_all[df_all["ticker"] == tk]
    for year in range(2018, 2026):
        dy = d[d["date"].dt.year == year].dropna(subset=["Close"])
        if len(dy) < 20: continue
        rows.append({
            "ticker":             tk,
            "ticker_clean":       tk.replace(".SA",""),
            "year":               year,
            "total_return_pct":   round(float(dy["cumulative_return"].iloc[-1]), 4),
            "avg_volatility_20d": round(float(dy["volatility_20d"].mean()), 4),
            "max_drawdown_pct":   round(float(((dy["Close"]/dy["Close"].cummax())-1).min()*100), 4),
            "avg_volume":         round(float(dy["Volume"].mean()), 0),
            "avg_amplitude_intraday": round(float(dy["amplitude_intraday"].mean()), 4),
        })

path_ano = os.path.join(OUTPUT_DIR, "ibovespa_summary_por_acao_por_ano.csv")
pd.DataFrame(rows).to_csv(path_ano, index=False)
print(f"💾 Resumo por ação × ano salvo: {path_ano}")

## Passo 10 — Taxa Selic histórica (API Banco Central)

Baixa duas séries do SGS (Sistema Gerenciador de Séries Temporais do BCB):

| Série | Descrição |
|-------|-----------|
| **432** | Meta da taxa Selic fixada pelo Copom (% a.a.) |
| **11** | Selic acumulada no período (% a.d.) — taxa diária efetiva |

O CSV de saída tem uma linha por dia útil e é pronto para merge com o histórico de ações no Observable.


In [ ]:
def fetch_bcb_serie(serie: int, start: str, end: str) -> pd.DataFrame:
    """
    Baixa uma série do SGS/BCB.
    start/end no formato 'YYYY-MM-DD'.
    Retorna DataFrame com colunas [date, valor].
    """
    d0 = pd.to_datetime(start).strftime("%d/%m/%Y")
    d1 = pd.to_datetime(end).strftime("%d/%m/%Y")
    url = (
        f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
        f"?formato=json&dataInicial={d0}&dataFinal={d1}"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    df["date"]  = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df[["date", "valor"]]


def fetch_selic_historica(start: str = START_DATE, end: str = END_DATE) -> pd.DataFrame:
    """
    Retorna DataFrame com colunas:
      date             — data (dia útil)
      selic_diaria_ad  — taxa Selic acumulada no período (% a.d.)
      selic_meta_aa    — meta Selic fixada pelo Copom (% a.a.)
    """
    print("🏦 Baixando Taxa Selic histórica via API do Banco Central...")

    # Série 11 — Selic acumulada no período (diária)
    df_diaria = fetch_bcb_serie(11, start, end)
    df_diaria = df_diaria.rename(columns={"valor": "selic_diaria_ad"})
    print(f"    Série 11 (Selic diária)    : {len(df_diaria):,} observações")

    # Série 432 — Meta Selic (Copom)
    df_meta = fetch_bcb_serie(432, start, end)
    df_meta = df_meta.rename(columns={"valor": "selic_meta_aa"})
    print(f"    Série 432 (Meta Selic a.a.): {len(df_meta):,} observações")

    # Merge — left join na série diária (mais granular)
    df_selic = pd.merge(df_diaria, df_meta, on="date", how="left")

    # Propaga a meta para os dias em que o Copom não se reuniu
    df_selic["selic_meta_aa"] = df_selic["selic_meta_aa"].ffill()

    df_selic = df_selic.sort_values("date").reset_index(drop=True)
    return df_selic


df_selic = fetch_selic_historica()

selic_path = os.path.join(OUTPUT_DIR, "selic_historica.csv")
df_selic.to_csv(selic_path, index=False)

print(f"\n Selic — amostra:")
print(df_selic.tail(10).to_string(index=False))
print(f"\n Selic histórica salva: {selic_path}")

## Passo 11 — Sensibilidade à Selic por setor (H3)

Gera uma tabela com o `beta` setorial em relação às variações da Selic após reuniões do Copom,
além de bases auxiliares para visualizações interativas no Observable.

In [ ]:
import statsmodels.api as sm

# 1. Identificar decisões do Copom (dias em que a meta Selic mudou)
selic = pd.read_csv(f"{OUTPUT_DIR}/selic_historica.csv", parse_dates=["date"])
selic = selic.sort_values("date").reset_index(drop=True)
selic["delta_selic"] = selic["selic_meta_aa"].diff()
copom_events = selic[selic["delta_selic"].abs() >= 0.25].copy()
print(f"Eventos Copom com variação >= 0.25 p.p.: {len(copom_events)}")

# 2. Carregar histórico e setor
setores_df = pd.read_csv(f"{OUTPUT_DIR}/ibovespa_setores.csv")
historico  = pd.read_csv(f"{OUTPUT_DIR}/ibovespa_historico_2018_2025.csv", parse_dates=["date"])
historico  = historico.merge(
    setores_df[["ticker_yf", "sector"]],
    left_on="ticker", right_on="ticker_yf", how="left"
)
historico["sector"] = historico["sector"].fillna("Unknown")
historico = historico.sort_values(["ticker", "date"])

# 3. Para cada evento Copom, calcular retorno dos 30 dias SEGUINTES por ação
# Janela começa no dia APÓS a decisão — sem leakage
WINDOW = 30

rows = []
for _, event in copom_events.iterrows():
    t0    = event["date"]
    t1    = t0 + pd.Timedelta(days=WINDOW)
    delta = event["delta_selic"]

    janela = historico[(historico["date"] > t0) & (historico["date"] <= t1)]
    if janela.empty:
        continue

    for (ticker, sector), g in janela.groupby(["ticker", "sector"]):
        if sector == "Unknown":
            continue
        g = g.sort_values("date")
        if len(g) < 10:
            continue
        ret = (g["Close"].iloc[-1] / g["Close"].iloc[0] - 1) * 100
        rows.append({
            "copom_date":  t0,
            "delta_selic": delta,
            "ticker":      ticker,
            "sector":      sector,
            "return_30d":  ret,
        })

df_reg = pd.DataFrame(rows)
print(f"Observações totais: {len(df_reg)}")
print(df_reg.groupby("sector").size().to_string())

# 4. Agregar por setor × evento (mediana entre ações do setor)
# Remove ruído idiossincrático de cada ação — isola comportamento setorial
df_agg = (
    df_reg
    .groupby(["copom_date", "delta_selic", "sector"])["return_30d"]
    .median()
    .reset_index()
)
print(f"\nObservações agregadas (setor × evento): {len(df_agg)}")

# 5. Regressão OLS por setor: retorno_mediano_setor ~ delta_selic
results = []
for sector, g in df_agg.groupby("sector"):
    if len(g) < 8:
        continue
    X = sm.add_constant(g["delta_selic"])
    y = g["return_30d"]
    model = sm.OLS(y, X).fit()
    results.append({
        "sector":      sector,
        "beta":        round(model.params["delta_selic"], 4),
        "pvalue":      round(model.pvalues["delta_selic"], 4),
        "r2":          round(model.rsquared, 4),
        "n_obs":       len(g),
        "significant": model.pvalues["delta_selic"] < 0.05,
    })

df_h3 = pd.DataFrame(results).sort_values("beta")
h3_path = os.path.join(OUTPUT_DIR, "h3_selic_beta_por_setor.csv")
df_h3.to_csv(h3_path, index=False)
print("\n", df_h3.to_string(index=False))

In [ ]:
# Confere quantas observações por setor
print(df_reg.groupby("sector")[["delta_selic", "return_30d"]].describe())

# E confere se delta_selic tem variância suficiente
print(df_reg["delta_selic"].value_counts().head(10))

## Passo 12 — Métricas trimestrais para H1, H2 e H4

Este bloco transforma o grafo trimestral em tabelas explícitas para testar as hipóteses:

- `H1`: colapso de correlação em crises
- `H2`: divergência setorial na recuperação
- `H4`: alinhamento entre comunidades detectadas e setores econômicos

In [ ]:
import numpy as np

def classify_phase(start):
    start = pd.to_datetime(start)
    if start < pd.Timestamp("2020-01-01"):
        return "pre_crise"
    if start <= pd.Timestamp("2020-06-30"):
        return "crise_covid"
    if start <= pd.Timestamp("2021-12-31"):
        return "recuperacao"
    if start <= pd.Timestamp("2022-12-31"):
        return "alta_selic"
    return "pos_recuperacao"

sector_lookup = (
    df_setores.assign(
        sector=lambda d: d["sector"].replace("", np.nan).fillna("Unknown")
    )
    .set_index("ticker_yf")["sector"]
    .to_dict()
)

def build_network_metrics(periods, df_logret, threshold=THRESHOLD):
    rows = []
    for name, p0, p1 in periods:
        sub = df_logret.loc[p0:p1].dropna(axis=1, thresh=30)
        tickers = sub.columns.tolist()
        if len(tickers) < 5:
            continue

        corr = sub.corr(method="pearson")
        mask = np.triu(np.ones(corr.shape, dtype=bool), 1)
        corr_pairs = (
            corr.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "source", "level_1": "target", 0: "correlation"})
        )
        corr_pairs["correlation"] = pd.to_numeric(corr_pairs["correlation"], errors="coerce")
        corr_pairs = corr_pairs.dropna(subset=["correlation"]).copy()
        if corr_pairs.empty:
            continue

        corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()
        corr_pairs["source_sector"] = corr_pairs["source"].map(lambda x: sector_lookup.get(x, "Unknown"))
        corr_pairs["target_sector"] = corr_pairs["target"].map(lambda x: sector_lookup.get(x, "Unknown"))
        corr_pairs["same_sector"] = (
            (corr_pairs["source_sector"] == corr_pairs["target_sector"])
            & (corr_pairs["source_sector"] != "Unknown")
        )

        thr_pairs = corr_pairs[corr_pairs["abs_correlation"] >= threshold].copy()
        possible_pairs = len(corr_pairs)
        density = len(thr_pairs) / possible_pairs if possible_pairs else np.nan

        G = nx.Graph()
        G.add_nodes_from(tickers)
        for row in thr_pairs.itertuples():
            G.add_edge(row.source, row.target, weight=float(row.abs_correlation))
        nx.set_node_attributes(G, {tk: sector_lookup.get(tk, "Unknown") for tk in tickers}, "sector")

        if G.number_of_edges() > 0:
            largest_component_share = (
                max(len(component) for component in nx.connected_components(G)) / len(tickers)
            )
            try:
                sector_assortativity = nx.attribute_assortativity_coefficient(G, "sector")
            except Exception:
                sector_assortativity = np.nan
        else:
            largest_component_share = np.nan
            sector_assortativity = np.nan

        rows.append({
            "period": name,
            "start": p0,
            "end": p1,
            "phase": classify_phase(p0),
            "n_nodes": len(tickers),
            "possible_pairs": possible_pairs,
            "n_edges": len(thr_pairs),
            "density": round(float(density), 6) if pd.notna(density) else np.nan,
            "avg_corr": round(float(corr_pairs["correlation"].mean()), 6),
            "avg_abs_corr": round(float(corr_pairs["abs_correlation"].mean()), 6),
            "median_abs_corr": round(float(corr_pairs["abs_correlation"].median()), 6),
            "p90_abs_corr": round(float(corr_pairs["abs_correlation"].quantile(0.9)), 6),
            "positive_edge_share": round(float((thr_pairs["correlation"] > 0).mean()), 6) if len(thr_pairs) else np.nan,
            "negative_edge_share": round(float((thr_pairs["correlation"] < 0).mean()), 6) if len(thr_pairs) else np.nan,
            "intra_sector_share": round(float(thr_pairs["same_sector"].mean()), 6) if len(thr_pairs) else np.nan,
            "inter_sector_share": round(float(1 - thr_pairs["same_sector"].mean()), 6) if len(thr_pairs) else np.nan,
            "largest_component_share": round(float(largest_component_share), 6) if pd.notna(largest_component_share) else np.nan,
            "sector_assortativity": round(float(sector_assortativity), 6) if pd.notna(sector_assortativity) else np.nan,
        })

    return pd.DataFrame(rows)

df_hyp_quarterly = build_network_metrics(PERIODS_QUARTERLY, df_logret)
df_hyp_thematic = build_network_metrics(PERIODS, df_logret)

path_hyp_q = os.path.join(OUTPUT_DIR, "h1_h2_network_metrics_quarterly.csv")
path_hyp_t = os.path.join(OUTPUT_DIR, "h1_h2_network_metrics_thematic.csv")
df_hyp_quarterly.to_csv(path_hyp_q, index=False)
df_hyp_thematic.to_csv(path_hyp_t, index=False)

print(f"💾 H1/H2 trimestral salvo: {path_hyp_q}")
print(f"💾 H1/H2 temático salvo : {path_hyp_t}")
print(df_hyp_quarterly.head().to_string(index=False))

## Passo 13 — Base longa para heatmap de comunidades (H4)

Exporta duas tabelas auxiliares:

- `h4_node_membership_quarterly.csv`: um nó por linha com comunidade e setor
- `h4_community_sector_heatmap.csv`: composição `comunidade × setor` por trimestre

In [ ]:
node_rows = []
heatmap_rows = []

for g in graph_quarterly:
    nodes_df = pd.DataFrame(g["nodes"])
    if nodes_df.empty:
        continue

    keep_cols = [c for c in [
        "id", "ticker_clean", "sector", "industry", "community",
        "avg_volume", "avg_volatility", "total_return"
    ] if c in nodes_df.columns]

    slim = nodes_df[keep_cols].copy()
    slim["period"] = g["period"]
    slim["start"] = g["start"]
    slim["end"] = g["end"]
    node_rows.append(slim)

    vis = slim[(slim["community"].notna()) & (slim["community"] >= 0)].copy()
    if vis.empty:
        continue

    community_sizes = vis.groupby("community").size().rename("community_size")
    sector_sizes = vis.groupby("sector").size().rename("sector_size")

    heat = (
        vis.groupby(["community", "sector"])
        .size()
        .rename("n_tickers")
        .reset_index()
        .merge(community_sizes.reset_index(), on="community", how="left")
        .merge(sector_sizes.reset_index(), on="sector", how="left")
    )
    heat["share_community"] = (heat["n_tickers"] / heat["community_size"]).round(6)
    heat["share_sector"] = (heat["n_tickers"] / heat["sector_size"]).round(6)
    heat["period"] = g["period"]
    heat["start"] = g["start"]
    heat["end"] = g["end"]
    heatmap_rows.append(heat)

df_h4_nodes = pd.concat(node_rows, ignore_index=True)
df_h4_heatmap = pd.concat(heatmap_rows, ignore_index=True)

h4_nodes_path = os.path.join(OUTPUT_DIR, "h4_node_membership_quarterly.csv")
h4_heatmap_path = os.path.join(OUTPUT_DIR, "h4_community_sector_heatmap.csv")

df_h4_nodes.to_csv(h4_nodes_path, index=False)
df_h4_heatmap.to_csv(h4_heatmap_path, index=False)

print(f"💾 Nós trimestrais H4     : {h4_nodes_path}")
print(f"💾 Heatmap comunidade/setor: {h4_heatmap_path}")
print(df_h4_heatmap.head().to_string(index=False))

## Passo 14 — Bases enriquecidas para H3

Cria versões prontas para o Observable:

- `h3_selic_beta_por_setor_enriched.csv`
- `h3_copom_event_sector_returns.csv`
- `h3_selic_beta_por_grupo.csv`
- `copom_events.csv`

In [ ]:
def classify_thesis_group(sector):
    s = str(sector or "").strip().lower()
    if any(key in s for key in ["financial", "bank", "real estate"]):
        return "sensivel_juros"
    if any(key in s for key in ["basic materials", "energy", "oil", "gas"]):
        return "commodities"
    return "outros"

df_h3_enriched = df_h3.copy()
df_h3_enriched["sector_group"] = df_h3_enriched["sector"].apply(classify_thesis_group)
df_h3_enriched["beta_abs"] = df_h3_enriched["beta"].abs().round(4)
df_h3_enriched["effect_direction"] = np.where(
    df_h3_enriched["beta"] < 0, "mais_sensivel_queda", "mais_sensivel_alta"
)
df_h3_enriched["sig_label"] = np.where(
    df_h3_enriched["significant"], "significativo", "nao_significativo"
)

df_h3_events = df_agg.copy()
df_h3_events["sector_group"] = df_h3_events["sector"].apply(classify_thesis_group)
df_h3_events["copom_direction"] = np.where(df_h3_events["delta_selic"] > 0, "alta", "queda")
df_h3_events = df_h3_events.rename(columns={"return_30d": "median_return_30d"})

group_results = []
for group_name, g in df_h3_events.groupby("sector_group"):
    if len(g) < 8 or g["delta_selic"].nunique() < 2:
        continue
    X = sm.add_constant(g["delta_selic"])
    y = g["median_return_30d"]
    model = sm.OLS(y, X).fit()
    group_results.append({
        "sector_group": group_name,
        "beta": round(float(model.params["delta_selic"]), 4),
        "pvalue": round(float(model.pvalues["delta_selic"]), 4),
        "r2": round(float(model.rsquared), 4),
        "n_obs": int(len(g)),
        "significant": bool(model.pvalues["delta_selic"] < 0.05),
    })

df_h3_group = pd.DataFrame(group_results).sort_values("beta")
df_copom = copom_events[["date", "delta_selic", "selic_meta_aa"]].rename(columns={"date": "copom_date"})

h3_enriched_path = os.path.join(OUTPUT_DIR, "h3_selic_beta_por_setor_enriched.csv")
h3_events_path = os.path.join(OUTPUT_DIR, "h3_copom_event_sector_returns.csv")
h3_group_path = os.path.join(OUTPUT_DIR, "h3_selic_beta_por_grupo.csv")
copom_path = os.path.join(OUTPUT_DIR, "copom_events.csv")

df_h3_enriched.to_csv(h3_enriched_path, index=False)
df_h3_events.to_csv(h3_events_path, index=False)
df_h3_group.to_csv(h3_group_path, index=False)
df_copom.to_csv(copom_path, index=False)

print(f"💾 H3 por setor enriquecido: {h3_enriched_path}")
print(f"💾 H3 eventos Copom       : {h3_events_path}")
print(f"💾 H3 por grupo           : {h3_group_path}")
print(f"💾 Eventos Copom          : {copom_path}")
print(df_h3_enriched.to_string(index=False))

## Resumo final — arquivos extras para as hipóteses

In [ ]:
hypothesis_files = {
    "h1_h2_network_metrics_quarterly.csv": "Métricas trimestrais para H1/H2 (densidade, correlação média, intra/inter-setor etc.)",
    "h1_h2_network_metrics_thematic.csv": "Métricas por janelas temáticas (pré-COVID, crise, recuperação, alta Selic...)",
    "h4_node_membership_quarterly.csv": "Atribuição de comunidade por ação e trimestre",
    "h4_community_sector_heatmap.csv": "Composição comunidade × setor por trimestre",
    "h3_selic_beta_por_setor_enriched.csv": "Beta setorial da Selic com classificação por grupo de tese",
    "h3_copom_event_sector_returns.csv": "Retornos medianos por setor após reuniões do Copom",
    "h3_selic_beta_por_grupo.csv": "Comparação agregada entre grupos sensíveis a juros e commodities",
    "copom_events.csv": "Lista de eventos Copom usados na hipótese H3",
}

print("=" * 72)
print(" ARQUIVOS EXTRAS PRONTOS PARA TESTAR H1–H4 NO OBSERVABLE")
print("=" * 72)
for nome, desc in hypothesis_files.items():
    p = os.path.join(OUTPUT_DIR, nome)
    status = f"({os.path.getsize(p)/1024:.0f} KB)" if os.path.exists(p) else "(ainda não gerado)"
    print(f"\n  {nome} {status}")
    print(f"     {desc}")